# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library. Following the Croissant schema standard, it provides a hands-on pathway to access, review, process, and visualize the dataset for clinicopathological analysis.

### Dataset Source
The dataset is described and structured following the Croissant schema.
- Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load the Croissant metadata and available record sets for analysis with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and create Dataset object
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata (title and description)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id`s.

Below we list the record sets, and for each, the available fields and their identifiers. This helps you select what part of the dataset to extract for further analysis.

In [ ]:
# Get all record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
We will extract data from all main record sets. Data is loaded into pandas DataFrames, using the Croissant `@id` of each record set and its fields.

In [ ]:
# Prepare extraction from all main record sets
dataframes = {}
for rs in record_sets:
    # Use record set @id for access as required
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded '{rs.name}' (@id: {rs.id}), records: {len(df)}, columns: {list(df.columns)[:10]}{'...' if len(df.columns) > 10 else ''}")

# Select the first record set as an example for further analysis
if record_sets:
    main_record_set = record_sets[0]
    main_rs_id = main_record_set.id
    df_main = dataframes[main_rs_id]
    print(f"\nFields in '{main_record_set.name}' (@id: {main_rs_id}):\n", list(df_main.columns))
    display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Apply some data transformations, filtering, and grouping. Each field is referenced by its Croissant `@id`.

Below, we choose a numeric field and a categorical (grouping) field, referencing their `@id`, and demonstrate normalization and grouping. Update the field `@id`s below to target the fields most relevant to your analysis.

In [ ]:
# -- Specify the field @id for numeric and grouping analysis --
# Replace these with field IDs appropriate for the selected record set.
# For demonstration, try to guess plausible field @id names, otherwise print all columns in previous cell and edit accordingly.
# Suppose '@id: https://api.app.sen.science/frontiers/7862866/age' for age and '@id: https://api.app.sen.science/frontiers/7862866/sex' for sex.

numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/age'    # e.g. patient age
group_field_id = 'https://api.app.sen.science/frontiers/7862866/sex'      # e.g. patient sex

# Verify fields exist
cols = df_main.columns.tolist()
print("Fields available in dataframe:", cols)
if numeric_field_id in cols and group_field_id in cols:
    # Filter for age > 50 as an example
    threshold = 50
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold}, count: {len(filtered_df)}")
    display(filtered_df[[numeric_field_id, group_field_id]].head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

    # Group by the group/categorical field
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count', 'std'])
    print(f"\nGrouped by {group_field_id}:")
    display(grouped_df)
else:
    print("At least one of the field IDs not found in the columns. Please check the available field IDs and update them above.")

## 5. Visualization
Visualize numeric field distribution and group comparisons. All visualizations will use the Croissant field `@id` to ensure traceability to the data model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if data available
if numeric_field_id in df_main.columns and group_field_id in df_main.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot comparison by group field
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df_main, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("Visualization skipped because required field(s) not found in data. Check field @id assignments.")

## 6. Conclusion

- You have loaded, reviewed, and explored the FAIR² Croissant-structured dataset using the `mlcroissant` library.
- Main dataset entities (record sets, fields, and columns) have been referenced and manipulated strictly by their Croissant `@id`, ensuring compliance with the schema.
- You saw how to extract, filter, group, normalize, and visualize fields for preliminary analysis.

> Next steps: refer to the output of the *Data Overview* section to use field IDs for your specific needs; update EDA and visualization as your data science questions evolve.